# Step 4：動力学の局所一次近似とQ関数と局所二次近似

## 概要

iLQRでは非線形システムの最適化問題を取り扱う。しかし、backward pass による入力修正則は線形・二次問題として解く必要がある。

そこで、基準軌道の近傍で、非線形動力学モデルを一次近似し、コスト関数を二次近似し、線形・二次問題を構成できるようにする。

## 1. 非線形動力学モデルの局所一次近似

具体的に、非線形バネモデルを対象として、基準軌道の近傍の一次近似を行う。

#### 状態方程式

非線形バネモデルの運動方程式を以下とする。

$$
m \ddot{x} = u - k x - k_3 x^3
$$

状態を $X$ とする。

$$
X = \begin{bmatrix} x \\ v \end{bmatrix} , \quad v = \dot{x}
$$

すると、状態方程式は以下となる。

$$
\dot{X} = \begin{bmatrix} \dot{x} \\ \dot{v} \end{bmatrix} =
\begin{bmatrix}  v \\ \dfrac{1}{m}\left( u - k x - k_3 x^3 \right) \end{bmatrix}
$$

#### 離散時間状態方程式

iLQRが扱うのは離散時間モデルであるため、刻み時間を $\Delta t$ として、前進Euler法で離散化する。

$$
\begin{aligned}
x_{n+1} &= x_n + \Delta t v_n \\
v_{n+1} &= v_n + \dfrac{\Delta t}{m}\left(u_n - k x_n - k_3 x_n^3 \right)
\end{aligned}
$$

状態$X$でまとめると、離散時間状態方程式は以下となる。この式は$x_n^3$があるため非線形である。

$$
\begin{aligned}
X_{n+1} &= f(X_n , u_n) \\
\begin{bmatrix}
x_{n+1} \\
v_{n+1} 
\end{bmatrix} &=
\begin{bmatrix}
x_n + \Delta t v_n \\
v_n + \dfrac{\Delta t}{m}\left(u_n - k x_n - k_3 x_n^3 \right)
\end{bmatrix}
\end{aligned}
$$

#### 基準軌道の近傍点

入力列を使い、上記の離散時間状態方程式をrolloutすると、基準軌道を得ることが出来る。

$$
\begin{array}{c}
\bar{u}_0 , \bar{u}_1 , \cdots , \bar{u}_{N-1} \\
\downarrow \\
\bar{X}_0, \bar{X}_1, \cdots , \bar{X}_N
\end{array}
$$

ここで、有限ホライゾン上の時刻$n$の基準点は以下である。

$$
(\bar{X}_n, \bar{u}_n)
$$

この基準点から少し離れた点(近傍点)$(X_n ,u_n)$ を以下のように$\delta$を用いて表す。

$$
\begin{aligned}
X_n &= \bar{X}_n + \delta X_n \\
u_n &= \bar{u}_n + \delta u_n
\end{aligned}
$$

$\delta$ は基準軌道からの小さなズレ、すなわち摂動である。

$$
\begin{aligned}
\delta X_n &= X_n - \bar{X}_n \\
\delta u_n &= u_n - \bar{u}_n
\end{aligned}
$$

#### Taylor 展開による線形化

非線形動力学モデルの離散時間状態方程式は以下の式である。

$$
X_{n+1} = f(X_n , u_n)
$$

これに基準軌道の近傍点を代入すると、以下となる。

$$
X_{n+1} = f(\bar{X}_n + \delta X_n , \bar{u}_n + \delta u_n)
$$

基準点($\bar{X}_n, \bar{u}_n$)の周りでTaylor展開し、一次近似の項までで構成すると、以下のように書ける。

$$
X_{n+1} \approx f(\bar{X}_n , \bar{u}_n ) + A_n \delta X_n + B_n \delta u_n
$$

$$
\begin{aligned}
A_n &= \left. \frac{\partial f}{\partial X} \right|_{(\bar{X}_n, \bar{u}_n)} \\
B_n &= \left. \frac{\partial f}{\partial u} \right|_{(\bar{X}_n, \bar{u}_n)} 
\end{aligned}
$$

基準軌道$\bar{X}_{n+1}$ は以下のように得ることが出来るので、

$$
\bar{X}_{n+1} =  f(\bar{X}_n , \bar{u}_n ) 
$$

基準軌道の近傍点は次の一次近似式で表すことが出来る。

$$
X_{n+1} \approx \bar{X}_{n+1} + A_n \delta X_n + B_n \delta u_n
$$

ここから、基準軌道からの小さなズレである摂動を、以下のように表すことが出来る。

$$
\delta {X}_{n+1} = X_{n+1} - \bar{X}_{n+1} \approx  A_n \delta X_n + B_n \delta u_n
$$

この式がiLQRで使う**局所**線形動力学摂動モデルである。<br>
「局所」とは基準点 $(\bar{X}_n, \bar{u}_n)$の近傍でこの一次式が有効であることを意味している。式は次状態そのものではなく、基準出力$\bar{X}_{n+1}$からのずれを表す。

$$
\delta {X}_{n+1} \approx A_n \delta X_n + B_n \delta u_n
$$

実際に$\delta X_n = \delta u_n = 0$であるならば、下記の基準点の近傍を表す一次近似式は$X_{n+1} = \bar{X}_{n+1}$となり、基準点からのズレがない場合、その点は基準点と一致するということになる。つまり$\delta {X}_{n+1}$は基準点からの摂動/ズレであることを示している。

$$
X_{n+1} \approx \bar{X}_{n+1} + A_n \delta X_n + B_n \delta u_n
$$

ここまでの導出で示した式を示すと、

- 非線形動力学モデルで計算する基準点
    - $\bar{X}_{n+1} = f(\bar{X}_n , \bar{u}_n)$
- 非線形動力学モデルに摂動を与えたときの基準点からのズレた点
    - ${X}_{n+1} = f(\bar{X}_n + \delta X_n , \bar{u}_n + \delta u_n)$
- 非線形動力学モデルの基準点からズレた点を、基準点周りでTaylor展開して一次近似で表した点
    - $X_{n+1}^{lin} = \bar{X}_{n+1} + A_n \delta X_n + B_n \delta u_n$
- 一次近似式で表した点において、基準点からのズレを表す式
    - $\delta {X}_{n+1} \approx A_n \delta X_n + B_n \delta u_n$
- 非線形動力学モデルの基準点と摂動を与えたときの基準点からのズレた点の差
    - $\delta {X}_{n+1} = X_{n+1} - \bar{X}_{n+1} \$





図で説明すると以下のようなものである。これは、$X_n$はベクトルであるが、それを1方向に切り出して示した模式図で、摂動($\delta X_n, \delta u_n$)を与えたときの非線形動力学モデルとその一次近似式の値、そして、$\delta X_{n+1}$と$A_n \delta X_n + B_n \delta u_n$ の関係を示している。<br>
($\delta X_n, \delta u_n$)が摂動が大きくなると、$\delta X_{n+1}$と$A_n \delta X_n + B_n \delta u_n$の差は大きくなり、摂動が小さいときは、その差は小さくなる。よって、一次近似式で非線形動力学モデルの挙動を模擬するには基準点に近い場所、つまり近傍点である必要がある。

<p align="center">
<img src="./images/local_linerlization.png" style="width:70%">
</p>
